# IaC & Container Orchestration

[Notebook 12](/courses/llm-eng/12-ai-infra.html) established the foundation: Terraform provisioning of ECR, ECS Fargate, and an ALB with TLS; plain Kubernetes manifests with rolling updates and CPU-based HPA; a Helm chart with `values.yaml` overrides per environment; and a GitHub Actions OIDC pipeline that pushes images and triggers ECS rolling deploys. This deep dive extends each of those foundations in directions that matter for real AI/ML platforms: Terraform modules and workspaces for multi-environment reuse, EKS provisioning with GPU node groups, KEDA event-driven autoscaling on SQS queue depth, advanced Helm authoring including hooks and library charts, GitOps reconciliation with ArgoCD, multi-tenant namespace isolation with resource quotas and network policies, and cost governance patterns for GPU-heavy workloads using spot instances, Karpenter, and Kubecost. Two additional sections close gaps from the main-path coverage audit: **S3 as an AI data store** (bucket layout, versioning, lifecycle rules, presigned URLs, and a `boto3`-backed `AIDataStore` class tested with `moto`), and **OpenTelemetry and Prometheus** (distributed tracing instrumentation for FastAPI, Prometheus Golden Signals for LLM services, and Grafana PromQL queries and alerting rules).

Setup:

In [ ]:
#| echo: false
import os, json
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel
from typing import Optional, Type

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient()

## Terraform Modules and Workspaces

**Why modules.** The `main.tf` from [notebook 12](/courses/llm-eng/12-ai-infra.html) is a flat file: it defines an ECR repository, a Secrets Manager secret, and an ECS cluster all in one place. That works for a single service, but an AI inference platform at a fintech firm runs a dozen services (compliance reviewer, risk scorer, trade summariser, document classifier) all sharing the same structural pattern: ECR repo, ECS Fargate service, ALB target group, Secrets Manager secret, CloudWatch log group. Duplicating 200 lines of HCL twelve times is how configuration drift begins.

<br>

**Module anatomy.** A Terraform module is a directory of `.tf` files with a defined input/output contract: `variables.tf` declares the inputs, `outputs.tf` declares what the module exposes to its caller, and `main.tf` contains the resources. The caller instantiates the module with a `module` block, passing values for the declared variables. The directory structure for an AI inference platform module looks like:

```
infra/
├── main.tf                    # root config — calls modules
├── variables.tf
├── terraform.tfvars           # dev defaults
├── terraform.prod.tfvars      # production overrides
└── modules/
    ├── ai-inference/
    │   ├── main.tf            # ECS service + task definition
    │   ├── variables.tf
    │   └── outputs.tf
    └── vpc/
        ├── main.tf
        ├── variables.tf
        └── outputs.tf
```

<br>

**Workspaces.** `terraform workspace` creates isolated state namespaces within a single configuration. Running `terraform workspace new staging` creates a separate state file (`terraform.tfstate.d/staging/terraform.tfstate`) so dev, staging, and prod can be managed from the same HCL without any risk of one environment's state corrupting another. The current workspace name is available as `terraform.workspace` inside HCL, allowing per-environment sizing without separate variable files.

The `modules/ai-inference/main.tf` module wrapping an ECS Fargate service as a reusable component:

In [ ]:
TERRAFORM_AI_INFERENCE_MODULE = """
# modules/ai-inference/variables.tf
variable "service_name"      { description = "ECS service name, e.g. compliance-reviewer" }
variable "image_uri"         { description = "Full ECR image URI including tag" }
variable "cpu"               { default = 512  }
variable "memory"            { default = 1024 }
variable "min_capacity"      { default = 1    }
variable "max_capacity"      { default = 10   }
variable "ecs_cluster_arn"   {}
variable "private_subnet_ids" { type = list(string) }
variable "security_group_id" {}
variable "execution_role_arn"{}
variable "secret_arns"       { type = list(string); default = [] }  # <1>
variable "environment_vars"  { type = map(string);  default = {} }

# modules/ai-inference/main.tf
resource "aws_ecr_repository" "svc" {
  name                 = var.service_name
  image_tag_mutability = "IMMUTABLE"                          # <2>
  image_scanning_configuration { scan_on_push = true }
}

resource "aws_cloudwatch_log_group" "svc" {
  name              = "/ecs/${var.service_name}"
  retention_in_days = 30
}

resource "aws_ecs_task_definition" "svc" {
  family                   = var.service_name
  requires_compatibilities = ["FARGATE"]
  network_mode             = "awsvpc"
  cpu                      = var.cpu
  memory                   = var.memory
  execution_role_arn       = var.execution_role_arn

  container_definitions = jsonencode([{
    name  = "app"
    image = var.image_uri
    portMappings = [{ containerPort = 8000, protocol = "tcp" }]

    secrets = [                                                # <3>
      for arn in var.secret_arns : {
        name      = split("/", arn)[length(split("/", arn)) - 1]
        valueFrom = arn
      }
    ]

    environment = [                                            # <4>
      for k, v in var.environment_vars : { name = k, value = v }
    ]

    logConfiguration = {
      logDriver = "awslogs"
      options = {
        "awslogs-group"         = aws_cloudwatch_log_group.svc.name
        "awslogs-region"        = "us-east-1"
        "awslogs-stream-prefix" = "ecs"
      }
    }
  }])
}

resource "aws_ecs_service" "svc" {
  name            = var.service_name
  cluster         = var.ecs_cluster_arn
  task_definition = aws_ecs_task_definition.svc.arn
  desired_count   = var.min_capacity
  launch_type     = "FARGATE"

  network_configuration {
    subnets          = var.private_subnet_ids
    security_groups  = [var.security_group_id]
    assign_public_ip = false
  }

  lifecycle {
    ignore_changes = [desired_count]                          # <5>
  }
}

resource "aws_appautoscaling_target" "svc" {
  max_capacity       = var.max_capacity
  min_capacity       = var.min_capacity
  resource_id        = "service/${split("/", var.ecs_cluster_arn)[1]}/${var.service_name}"
  scalable_dimension = "ecs:service:DesiredCount"
  service_namespace  = "ecs"
}

# modules/ai-inference/outputs.tf
output "ecr_repository_url" { value = aws_ecr_repository.svc.repository_url }
output "ecs_service_name"   { value = aws_ecs_service.svc.name }
output "log_group_name"     { value = aws_cloudwatch_log_group.svc.name }
"""

print(TERRAFORM_AI_INFERENCE_MODULE)

1. `secret_arns` accepts a list of Secrets Manager ARNs. The `for` expression inside `container_definitions` converts each ARN into a `{ name, valueFrom }` pair — the secret name is derived by splitting on `/` and taking the last segment.
2. `IMMUTABLE` image tags mean that once a digest is pushed under a tag, it cannot be overwritten. This is a fintech-grade control: deployed images are forensically reproducible and cannot be silently replaced.
3. The `secrets` list injects Secrets Manager values as environment variables at container start time — the values are never stored in the task definition JSON or in version control.
4. `environment_vars` is a plain `map(string)` for non-sensitive configuration (e.g. `LANGFUSE_HOST`, feature flags). Sensitive values always go through `secret_arns`.
5. `ignore_changes = [desired_count]` prevents Terraform from resetting the replica count after an autoscaler event has scaled the service up or down — without this, every `terraform apply` would reset to `min_capacity`.

The root `main.tf` calling the module for two services, and the per-environment `tfvars` files:

In [ ]:
TERRAFORM_ROOT_MAIN = """
# infra/main.tf  —  root configuration calling the ai-inference module
terraform {
  required_providers {
    aws = { source = "hashicorp/aws", version = "~> 5.0" }
  }
  backend "s3" {
    bucket         = "fintech-tf-state"
    key            = "ai-platform/${terraform.workspace}/terraform.tfstate"  # <1>
    region         = "us-east-1"
    dynamodb_table = "fintech-tf-lock"                                        # <2>
    encrypt        = true
  }
}

locals {
  env = terraform.workspace  # "dev" | "staging" | "prod"
  tags = {
    Environment = local.env
    ManagedBy   = "terraform"
    CostCenter  = "ai-platform"
  }
}

module "vpc" {
  source   = "./modules/vpc"
  env      = local.env
  cidr     = var.vpc_cidr
}

module "compliance_reviewer" {
  source              = "./modules/ai-inference"           # <3>
  service_name        = "compliance-reviewer-${local.env}"
  image_uri           = "${var.ecr_account}.dkr.ecr.us-east-1.amazonaws.com/compliance-reviewer:${var.image_tag}"
  cpu                 = var.compliance_cpu
  memory              = var.compliance_memory
  min_capacity        = var.compliance_min_replicas
  max_capacity        = var.compliance_max_replicas
  ecs_cluster_arn     = aws_ecs_cluster.main.arn
  private_subnet_ids  = module.vpc.private_subnet_ids
  security_group_id   = aws_security_group.ecs_tasks.id
  execution_role_arn  = aws_iam_role.ecs_exec.arn
  secret_arns         = [aws_secretsmanager_secret.openai.arn]
  environment_vars    = { LANGFUSE_HOST = "http://langfuse:3000" }
}

module "risk_scorer" {
  source              = "./modules/ai-inference"
  service_name        = "risk-scorer-${local.env}"
  image_uri           = "${var.ecr_account}.dkr.ecr.us-east-1.amazonaws.com/risk-scorer:${var.image_tag}"
  cpu                 = var.risk_cpu
  memory              = var.risk_memory
  min_capacity        = var.risk_min_replicas
  max_capacity        = var.risk_max_replicas
  ecs_cluster_arn     = aws_ecs_cluster.main.arn
  private_subnet_ids  = module.vpc.private_subnet_ids
  security_group_id   = aws_security_group.ecs_tasks.id
  execution_role_arn  = aws_iam_role.ecs_exec.arn
  secret_arns         = [aws_secretsmanager_secret.openai.arn]
}

# infra/terraform.tfvars  (dev defaults — checked in)
# compliance_cpu           = 512
# compliance_memory        = 1024
# compliance_min_replicas  = 1
# compliance_max_replicas  = 4
# risk_cpu                 = 256
# risk_memory              = 512
# risk_min_replicas        = 1
# risk_max_replicas        = 3

# infra/terraform.prod.tfvars  (production — checked in, no secrets)
# compliance_cpu           = 2048
# compliance_memory        = 4096
# compliance_min_replicas  = 3
# compliance_max_replicas  = 20
# risk_cpu                 = 1024
# risk_memory              = 2048
# risk_min_replicas        = 2
# risk_max_replicas        = 10

# Workspace commands:
# terraform workspace new prod
# terraform workspace select prod
# terraform apply -var-file=terraform.prod.tfvars          # <4>
"""

print(TERRAFORM_ROOT_MAIN)

1. The S3 key includes `${terraform.workspace}` so each workspace writes to an isolated state path — `ai-platform/dev/terraform.tfstate`, `ai-platform/prod/terraform.tfstate`. They share the same bucket but are entirely independent state files.
2. The `dynamodb_table` parameter enables state locking: only one `terraform apply` can hold the lock at a time, preventing concurrent runs from corrupting the state file. The DynamoDB table is itself provisioned once by a separate bootstrap configuration.
3. Both services call the same `./modules/ai-inference` source path. If the module's interface changes — say we add a `health_check_path` variable — that change propagates to every caller in a single PR.
4. Combining workspaces with environment-specific `tfvars` files is the recommended multi-environment pattern: workspaces isolate state, `tfvars` files isolate configuration values. Secrets are never in `tfvars` — they are fetched from Secrets Manager at plan/apply time via `data.aws_secretsmanager_secret_version`.

## EKS Cluster Provisioning for AI Workloads

**ECS Fargate vs. EKS for ML.** ECS Fargate is the right choice for stateless inference services with predictable resource profiles — the compliance reviewer from [notebook 12](/courses/llm-eng/12-ai-infra.html) fits this pattern. ML platforms diverge from this in three ways that push teams toward EKS: (1) **GPU workloads** — Fargate does not support NVIDIA GPUs; only EC2-backed ECS or EKS node groups can attach GPU devices. (2) **Event-driven autoscaling** — KEDA (covered in the next section) extends Kubernetes HPA with external metric sources like SQS queue depth; there is no equivalent on ECS. (3) **Custom schedulers** — batch inference jobs, feature pipelines, and fine-tuning runs have heterogeneous scheduling requirements (topology-aware placement, preemption, gang scheduling) that Kubernetes batch frameworks like Volcano handle natively.

<br>

**GPU node group taints.** GPU nodes carry a Kubernetes taint `nvidia.com/gpu=true:NoSchedule`. This prevents CPU-only pods from landing on GPU nodes and wasting expensive GPU capacity. Only pods with a matching **toleration** in their spec — and a GPU resource request (`nvidia.com/gpu: 1`) — will be scheduled onto GPU nodes. This is the primary mechanism for ensuring GPU nodes are used exclusively by AI workloads.

The complete Terraform configuration for an EKS cluster with CPU and GPU node groups:

In [ ]:
TERRAFORM_EKS = """
# infra/eks.tf

# ── IAM role for the EKS control plane ────────────────────────────────────────
resource "aws_iam_role" "eks_cluster" {
  name = "eks-cluster-role"
  assume_role_policy = jsonencode({
    Version = "2012-10-17"
    Statement = [{
      Effect    = "Allow"
      Principal = { Service = "eks.amazonaws.com" }
      Action    = "sts:AssumeRole"
    }]
  })
}
resource "aws_iam_role_policy_attachment" "eks_cluster_policy" {
  role       = aws_iam_role.eks_cluster.name
  policy_arn = "arn:aws:iam::aws:policy/AmazonEKSClusterPolicy"
}

# ── EKS cluster with OIDC provider ────────────────────────────────────────────
resource "aws_eks_cluster" "ml_platform" {
  name     = "ml-platform-${var.env}"
  role_arn = aws_iam_role.eks_cluster.arn
  version  = "1.30"

  vpc_config {
    subnet_ids              = concat(var.private_subnet_ids, var.public_subnet_ids)
    endpoint_private_access = true
    endpoint_public_access  = true                               # <1>
    public_access_cidrs     = ["10.0.0.0/8"]                    # <2>
  }

  enabled_cluster_log_types = ["api", "audit", "authenticator"]  # <3>
  depends_on = [aws_iam_role_policy_attachment.eks_cluster_policy]
}

# OIDC provider — required for IAM Roles for Service Accounts (IRSA)
data "tls_certificate" "eks" {
  url = aws_eks_cluster.ml_platform.identity[0].oidc[0].issuer
}
resource "aws_iam_openid_connect_provider" "eks" {
  client_id_list  = ["sts.amazonaws.com"]
  thumbprint_list = [data.tls_certificate.eks.certificates[0].sha1_fingerprint]
  url             = aws_eks_cluster.ml_platform.identity[0].oidc[0].issuer
}

# ── IAM role for worker nodes ──────────────────────────────────────────────────
resource "aws_iam_role" "eks_node" {
  name = "eks-node-role"
  assume_role_policy = jsonencode({
    Version = "2012-10-17"
    Statement = [{
      Effect    = "Allow"
      Principal = { Service = "ec2.amazonaws.com" }
      Action    = "sts:AssumeRole"
    }]
  })
}
resource "aws_iam_role_policy_attachment" "node_worker" {
  role       = aws_iam_role.eks_node.name
  policy_arn = "arn:aws:iam::aws:policy/AmazonEKSWorkerNodePolicy"
}
resource "aws_iam_role_policy_attachment" "node_cni" {
  role       = aws_iam_role.eks_node.name
  policy_arn = "arn:aws:iam::aws:policy/AmazonEKS_CNI_Policy"
}
resource "aws_iam_role_policy_attachment" "node_ecr" {
  role       = aws_iam_role.eks_node.name
  policy_arn = "arn:aws:iam::aws:policy/AmazonEC2ContainerRegistryReadOnly"
}

# ── CPU inference node group (c5.xlarge) ──────────────────────────────────────
resource "aws_eks_node_group" "cpu_inference" {
  cluster_name    = aws_eks_cluster.ml_platform.name
  node_group_name = "cpu-inference"
  node_role_arn   = aws_iam_role.eks_node.arn
  subnet_ids      = var.private_subnet_ids
  instance_types  = ["c5.xlarge"]                              # <4>

  scaling_config {
    desired_size = 2
    min_size     = 1
    max_size     = 20
  }

  labels = { role = "cpu-inference", "node.kubernetes.io/instance-type" = "c5.xlarge" }
  depends_on = [
    aws_iam_role_policy_attachment.node_worker,
    aws_iam_role_policy_attachment.node_cni,
    aws_iam_role_policy_attachment.node_ecr,
  ]
}

# ── GPU inference node group (g4dn.xlarge, spot) ──────────────────────────────
resource "aws_eks_node_group" "gpu_inference" {
  cluster_name    = aws_eks_cluster.ml_platform.name
  node_group_name = "gpu-inference"
  node_role_arn   = aws_iam_role.eks_node.arn
  subnet_ids      = var.private_subnet_ids
  instance_types  = ["g4dn.xlarge", "g4dn.2xlarge"]           # <5>
  capacity_type   = "SPOT"                                      # <6>
  ami_type        = "AL2_x86_64_GPU"                           # <7>

  scaling_config {
    desired_size = 0
    min_size     = 0
    max_size     = 10
  }

  taint {
    key    = "nvidia.com/gpu"
    value  = "true"
    effect = "NO_SCHEDULE"                                      # <8>
  }

  labels = { role = "gpu-inference", "nvidia.com/gpu" = "true" }
  depends_on = [
    aws_iam_role_policy_attachment.node_worker,
    aws_iam_role_policy_attachment.node_cni,
    aws_iam_role_policy_attachment.node_ecr,
  ]
}

# ── EKS add-ons ───────────────────────────────────────────────────────────────
resource "aws_eks_addon" "coredns" {
  cluster_name = aws_eks_cluster.ml_platform.name
  addon_name   = "coredns"
}
resource "aws_eks_addon" "kube_proxy" {
  cluster_name = aws_eks_cluster.ml_platform.name
  addon_name   = "kube-proxy"
}
resource "aws_eks_addon" "vpc_cni" {
  cluster_name = aws_eks_cluster.ml_platform.name
  addon_name   = "vpc-cni"
}
"""

print(TERRAFORM_EKS)

1. `endpoint_public_access = true` with `public_access_cidrs` locked to the corporate VPN CIDR means `kubectl` access is restricted to users on the VPN — the API server is not open to the public internet.
2. The corporate VPN range `10.0.0.0/8` is an example. In practice this is a `variable` sourced from the network team's CMDB.
3. `audit` logs capture every Kubernetes API call including who called it and with what arguments — required for SOC 2 and PCI DSS audit trails. The logs land in CloudWatch Logs under `/aws/eks/ml-platform-prod/cluster`.
4. `c5.xlarge` (4 vCPU, 8 GiB RAM) is a cost-effective node for CPU inference — LLM proxy services, embedding servers, and rule-based classifiers that do not need GPU acceleration.
5. Multiple `instance_types` in the GPU group lets the EKS Managed Node Group use whichever g4dn variant has spot capacity available, reducing interruption probability.
6. `capacity_type = "SPOT"` cuts GPU node cost by 60–80% relative to on-demand. The tradeoff is interruption; we address this in the Cost Governance section.
7. `AL2_x86_64_GPU` is the Amazon Linux 2 AMI with NVIDIA drivers and the `nvidia-container-runtime` pre-installed. Without this, CUDA libraries are not available inside containers.
8. `NO_SCHEDULE` taint means no pod will be scheduled on a GPU node unless it explicitly **tolerates** `nvidia.com/gpu=true`. An AI inference pod that needs a GPU must include both the toleration and a `resources.requests: { nvidia.com/gpu: "1" }` field in its spec.

A GPU inference pod spec illustrating the required toleration and resource request:

In [ ]:
K8S_GPU_POD = """
# k8s/gpu-inference-deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: embeddings-server
  namespace: risk-ai
spec:
  replicas: 1
  selector:
    matchLabels:
      app: embeddings-server
  template:
    metadata:
      labels:
        app: embeddings-server
        team: risk
        service: embeddings-server
        environment: production
    spec:
      tolerations:                              # <1>
        - key: "nvidia.com/gpu"
          operator: "Equal"
          value: "true"
          effect: "NoSchedule"
      nodeSelector:
        role: gpu-inference                     # <2>
      containers:
        - name: server
          image: 123456789.dkr.ecr.us-east-1.amazonaws.com/embeddings-server:latest
          resources:
            requests:
              cpu: "2"
              memory: "8Gi"
              nvidia.com/gpu: "1"               # <3>
            limits:
              nvidia.com/gpu: "1"
          ports:
            - containerPort: 8080
          readinessProbe:
            httpGet:
              path: /health
              port: 8080
            initialDelaySeconds: 30             # <4>
            periodSeconds: 10
"""

print(K8S_GPU_POD)

1. The toleration matches the node group taint exactly — `key`, `value`, and `effect` must all match. A pod missing this toleration will remain `Pending` indefinitely if no non-tainted node can satisfy its GPU resource request.
2. `nodeSelector` pins the pod to the GPU node group via the label we set in the Terraform node group definition. Without this, a future non-GPU node that happens to have no taints could theoretically receive the pod.
3. `nvidia.com/gpu: "1"` is the Kubernetes extended resource request. The NVIDIA device plugin daemonset (installed separately via Helm) exposes the GPU as an allocatable resource and handles the CUDA device binding inside the container.
4. GPU model servers have long cold-start times — loading multi-gigabyte model weights from S3 can take 30–120 seconds. `initialDelaySeconds: 30` prevents the readiness probe from marking the pod unhealthy during model load.

## KEDA — Event-Driven Autoscaling

**HPA limitations for AI workloads.** The standard Kubernetes Horizontal Pod Autoscaler scales on CPU utilisation or memory — it answers the question "are the running pods busy?" For an asynchronous AI service like the compliance reviewer (which consumes documents from an SQS queue), CPU may be low even when 10,000 documents are waiting, simply because the current two pods are processing as fast as they can and CPU is not the bottleneck. HPA would not add replicas in this scenario. What we actually want is: "how many messages are queued, and how many replicas do we need to drain that backlog within our SLA window?"

<br>

**KEDA mechanics.** KEDA (Kubernetes Event-Driven Autoscaler) installs as a lightweight operator and extends HPA with a `ScaledObject` CRD that binds a deployment to one or more external **scalers**. Each scaler polls an external metric source (SQS, Kafka, RabbitMQ, Prometheus, CloudWatch, etc.) and reports a queue length. KEDA converts this queue length into a desired replica count using the formula:

$$r = \left\lceil \frac{q}{m} \right\rceil$$

where $q$ is the current queue depth and $m$ is the `targetValue` (messages per replica). KEDA also handles scale-to-zero: when $q = 0$, the deployment is scaled to zero replicas, which is impossible with HPA (minimum is 1). Scale-to-zero is critical for GPU workloads where idle nodes cost \$0.526/hr per g4dn.xlarge even without serving a single request.

The KEDA `TriggerAuthentication` and `ScaledObject` for the compliance reviewer SQS queue:

In [ ]:
KEDA_SCALED_OBJECT = """
# k8s/keda/trigger-auth.yaml
apiVersion: keda.sh/v1alpha1
kind: TriggerAuthentication
metadata:
  name: sqs-trigger-auth
  namespace: compliance-ai
spec:
  podIdentity:
    provider: aws                              # <1>
---
# k8s/keda/scaled-object.yaml
apiVersion: keda.sh/v1alpha1
kind: ScaledObject
metadata:
  name: compliance-reviewer-scaler
  namespace: compliance-ai
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: compliance-reviewer
  minReplicaCount: 0                          # <2>
  maxReplicaCount: 50
  pollingInterval: 15                         # <3>
  cooldownPeriod:  120
  triggers:
    - type: aws-sqs-queue
      authenticationRef:
        name: sqs-trigger-auth
      metadata:
        queueURL:       https://sqs.us-east-1.amazonaws.com/123456789/compliance-docs
        queueLength:    "10"                  # <4>
        awsRegion:      us-east-1
        identityOwner:  pod
"""

print(KEDA_SCALED_OBJECT)

1. `provider: aws` uses IRSA (IAM Roles for Service Accounts) — the KEDA operator assumes an IAM role bound to its Kubernetes service account. No AWS credentials are stored in Kubernetes secrets; the IAM role grants `sqs:GetQueueAttributes` on the target queue.
2. `minReplicaCount: 0` enables scale-to-zero. When the queue is empty, KEDA scales the deployment to zero pods. The next message arriving in the queue triggers a scale-up within one polling interval.
3. `pollingInterval: 15` means KEDA checks the SQS queue depth every 15 seconds. The `cooldownPeriod: 120` prevents rapid oscillation: after scaling down, at least 120 seconds must elapse before another scale-down event.
4. `queueLength: "10"` is $m$ in the formula $r = \lceil q / m \rceil$. With 100 messages queued, KEDA targets $\lceil 100 / 10 \rceil = 10$ replicas. With 1 message, it targets $\lceil 1 / 10 \rceil = 1$ replica.

A comparison of HPA and KEDA for AI service scaling decisions:

In [ ]:
HPA_VS_KEDA = [
    ("Metric source",     "CPU / memory (in-cluster)",          "Any external metric (SQS, Kafka, CloudWatch, Prometheus, …)"),
    ("Scale-to-zero",     "No (minimum 1 replica)",             "Yes — pods can be scaled to 0"),
    ("Scaling formula",   "currentMetricValue / targetValue",   "ceil(queueDepth / targetValue)"),
    ("Latency model",     "Reactive — scales after CPU spikes", "Proactive — scales before pods saturate"),
    ("GPU idle cost",     "Idle pods still run ($$)",           "Zero replicas → zero GPU node cost"),
    ("Best for",          "Synchronous HTTP services",          "Async queue consumers, batch inference, fine-tuning jobs"),
    ("Auth complexity",   "None (in-cluster metrics)",          "IRSA or secret-based per scaler"),
    ("Cold-start risk",   "Low (always ≥1 replica)",            "High — first message waits for pod start (mitigate with minReplicas=1 for latency-sensitive queues)"),
]

header = f"{'Dimension':<24} {'HPA':<42} {'KEDA'}"
print(header)
print("-" * len(header))
for dim, hpa, keda in HPA_VS_KEDA:
    print(f"{dim:<24} {hpa:<42} {keda}")

:::{.callout-tip}
For a compliance reviewer that must respond within 30 seconds of a document arriving in the queue, set `minReplicaCount: 1` to keep at least one warm pod. Reserve `minReplicaCount: 0` for batch workloads — nightly re-scoring jobs, periodic fine-tuning runs — where a 60-second cold start is acceptable and GPU idle cost reduction matters more.

:::

## Helm Chart Authoring — Advanced

**Where notebook 12 left off.** The Helm section in [notebook 12](/courses/llm-eng/12-ai-infra.html) covered the basic chart structure (`Chart.yaml`, `values.yaml`, `values.prod.yaml`, a templated `deployment.yaml`) and the `helm install/upgrade/rollback` commands. Four patterns matter at production scale that were not covered: (1) **chart dependencies** for pulling in shared subcharts (e.g. PostgreSQL); (2) **lifecycle hooks** for running database migrations or smoke tests as part of the release; (3) **library charts** for sharing template fragments across many service charts without duplication; and (4) **`helm test`** for automated post-deploy validation.

<br>

**Semantic versioning in CI.** Every Helm chart has two versions: `version` (the chart packaging version) and `appVersion` (the application Docker image tag). The convention in a CI/CD pipeline is to bump `version` on every build using the git commit count or a `PATCH` increment, and to set `appVersion` to `$GITHUB_SHA`. This means every PR that merges produces a uniquely versioned chart artifact stored in an OCI registry (e.g. ECR), and rolling back to any previous state is a one-line `helm rollback` or an ArgoCD sync to the previous chart digest.

A `Chart.yaml` with a PostgreSQL dependency and a complete conditional `templates/hpa.yaml`:

In [ ]:
HELM_ADVANCED_CHART = """
# charts/compliance-reviewer/Chart.yaml
apiVersion: v2
name: compliance-reviewer
description: Async document compliance review service with PostgreSQL audit log
type: application
version: 1.14.0          # bumped by CI on every merge to main         # <1>
appVersion: "a3f1b2c4"   # set to $GITHUB_SHA at build time

dependencies:                                                            # <2>
  - name: postgresql
    version: "15.5.x"
    repository: "https://charts.bitnami.com/bitnami"
    condition: postgresql.enabled

# charts/compliance-reviewer/templates/hpa.yaml
{{- if .Values.autoscaling.enabled }}
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: {{ include "compliance-reviewer.fullname" . }}
  namespace: {{ .Release.Namespace }}
  labels:
    {{- include "compliance-reviewer.labels" . | nindent 4 }}
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: {{ include "compliance-reviewer.fullname" . }}
  minReplicas: {{ .Values.autoscaling.minReplicas }}
  maxReplicas: {{ .Values.autoscaling.maxReplicas }}
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: {{ .Values.autoscaling.targetCPUUtilizationPercentage }}
    {{- if .Values.autoscaling.targetMemoryUtilizationPercentage }}
    - type: Resource
      resource:
        name: memory
        target:
          type: Utilization
          averageUtilization: {{ .Values.autoscaling.targetMemoryUtilizationPercentage }}
    {{- end }}
{{- end }}                                                               # <3>

# charts/compliance-reviewer/templates/migration-hook.yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: {{ include "compliance-reviewer.fullname" . }}-migrate
  annotations:
    "helm.sh/hook": pre-upgrade,pre-install                             # <4>
    "helm.sh/hook-weight": "-5"
    "helm.sh/hook-delete-policy": before-hook-creation,hook-succeeded  # <5>
spec:
  template:
    spec:
      restartPolicy: Never
      containers:
        - name: migrate
          image: {{ .Values.image.repository }}:{{ .Values.image.tag }}
          command: ["python", "-m", "alembic", "upgrade", "head"]
          envFrom:
            - secretRef:
                name: {{ .Values.secretRef }}

# charts/compliance-reviewer/templates/tests/test-inference.yaml
apiVersion: v1
kind: Pod
metadata:
  name: {{ include "compliance-reviewer.fullname" . }}-test
  annotations:
    "helm.sh/hook": test                                                 # <6>
spec:
  restartPolicy: Never
  containers:
    - name: test
      image: curlimages/curl:8.7.1
      command:
        - sh
        - -c
        - |
          STATUS=$(curl -s -o /dev/null -w "%{http_code}" \
            http://{{ include "compliance-reviewer.fullname" . }}/health)
          [ "$STATUS" = "200" ] && echo "PASS" || (echo "FAIL: $STATUS" && exit 1)
"""

print(HELM_ADVANCED_CHART)

1. CI bumps `version` by running `yq e '.version = "1.14.0"' -i Chart.yaml` (or similar) before `helm package`. The chart is then pushed to an OCI registry with `helm push compliance-reviewer-1.14.0.tgz oci://123456789.dkr.ecr.us-east-1.amazonaws.com/helm-charts`.
2. The `dependencies` block pulls the Bitnami PostgreSQL subchart at install time (`helm dependency update`). Setting `condition: postgresql.enabled` lets staging disable the subchart when using an external RDS instance, while local dev runs postgres in-cluster.
3. The `{{- if .Values.autoscaling.enabled }}` guard means the entire HPA resource is omitted from the rendered manifest if `autoscaling.enabled: false`. This keeps dev deployments simple: a single replica with no autoscaler.
4. `helm.sh/hook: pre-upgrade,pre-install` runs the migration Job *before* the new Deployment pods start. This guarantees the database schema matches the application code before any new pod accepts traffic — the classic forward-compatible migration pattern.
5. `hook-delete-policy: before-hook-creation,hook-succeeded` cleans up the Job after success. Without this, re-running `helm upgrade` a second time would fail because the Job resource already exists.
6. `helm.sh/hook: test` marks this Pod as a test fixture. Running `helm test compliance-reviewer` after an upgrade spins up this pod and exits 0 only if the pod exits 0 — a lightweight integration gate that can be added to the CI/CD pipeline after `helm upgrade`.

A Helm library chart for sharing label templates across all service charts in the platform:

In [ ]:
HELM_LIBRARY_CHART = """
# charts/fintech-common/Chart.yaml
apiVersion: v2
name: fintech-common
description: Shared Helm template library for the AI platform
type: library                                                  # <1>
version: 0.3.0

# charts/fintech-common/templates/_labels.tpl
{{- define "fintech-common.labels" -}}
app.kubernetes.io/name:       {{ .Chart.Name }}
app.kubernetes.io/instance:   {{ .Release.Name }}
app.kubernetes.io/version:    {{ .Chart.AppVersion | quote }}
app.kubernetes.io/managed-by: {{ .Release.Service }}
team:                         {{ .Values.team | default "platform" }}
environment:                  {{ .Values.environment | default .Release.Namespace }}
cost-center:                  ai-platform                     # <2>
{{- end }}

{{- define "fintech-common.securityContext" -}}
runAsNonRoot:             true                                 # <3>
runAsUser:                1000
readOnlyRootFilesystem:   true
allowPrivilegeEscalation: false
capabilities:
  drop: ["ALL"]
{{- end }}

# Usage in compliance-reviewer Chart.yaml:
# dependencies:
#   - name: fintech-common
#     version: "0.3.x"
#     repository: "oci://123456789.dkr.ecr.us-east-1.amazonaws.com/helm-charts"

# Usage in compliance-reviewer deployment.yaml:
# labels:
#   {{- include "fintech-common.labels" . | nindent 4 }}
# securityContext:
#   {{- include "fintech-common.securityContext" . | nindent 10 }}
"""

print(HELM_LIBRARY_CHART)

1. `type: library` marks the chart as a library — it contains only named templates (files beginning with `_`) and no installable resources. Attempting to `helm install` a library chart directly will fail. This enforces the pattern: library charts are consumed as dependencies, never deployed directly.
2. The `cost-center: ai-platform` label appears on every pod across all service charts that use the library. Kubecost (covered in the Cost Governance section) uses this label to aggregate spend across the entire AI platform without requiring teams to remember to add it per-service.
3. The `fintech-common.securityContext` template enforces the K8s hardening baseline from the Production Checklist: non-root user, read-only root filesystem, no privilege escalation, all capabilities dropped. Centralising this in the library means a security policy change is a one-line PR to `fintech-common`, not a 12-chart diff.

## GitOps with ArgoCD

**The push model and its audit gap.** The GitHub Actions pipeline from [notebook 12](/courses/llm-eng/12-ai-infra.html) is a **push-based** deployment model: the CI runner calls `aws ecs update-service` or `helm upgrade` directly, pushing a change to the cluster. The cluster state after the push is whatever the CI script produced. If someone then runs `kubectl apply -f` manually, or if a transient network failure during the pipeline leaves the cluster in a half-updated state, there is no mechanism to detect or correct the drift. In regulated fintech, an auditor asking "what was deployed at 14:37 UTC on the 3rd?" has to reconstruct the answer from CI logs — not from the cluster itself.

<br>

**The pull model.** GitOps inverts the flow: a Git repository is the **single source of truth** for desired cluster state. An in-cluster operator (ArgoCD) continuously **polls** the repository and reconciles the cluster toward the declared state. A human cannot deploy by running `kubectl apply` — any manual change is overwritten by ArgoCD within minutes. Every deployment is a Git commit: reviewable, revertible, and attributed to a named author. For fintech, this is the immutable audit trail requirement satisfied at the infrastructure layer.

<br>

**Promotion workflow.** With ArgoCD, promoting from staging to production is a pull request that changes `config/prod/values.yaml` (e.g. bumps the image tag). A compliance engineer approves the PR. The merge triggers ArgoCD to detect the diff and reconcile — no manual `kubectl` or `helm` commands are executed by a human. The full deployment history is the Git log of `config/`.

ArgoCD `Application` CRDs for dev, staging, and production environments pointing at the same Helm chart:

In [ ]:
ARGOCD_APPLICATIONS = """
# config/argocd/compliance-reviewer-dev.yaml
apiVersion: argoproj.io/v1alpha1
kind: Application
metadata:
  name: compliance-reviewer-dev
  namespace: argocd
spec:
  project: ai-platform
  source:
    repoURL: https://github.com/fintech-corp/ai-platform-config  # <1>
    targetRevision: main
    path: config/dev/compliance-reviewer
    helm:
      valueFiles:
        - values.yaml
  destination:
    server: https://kubernetes.default.svc
    namespace: compliance-ai
  syncPolicy:
    automated:                                                    # <2>
      prune: true
      selfHeal: true
    syncOptions:
      - CreateNamespace=true
---
# config/argocd/compliance-reviewer-prod.yaml
apiVersion: argoproj.io/v1alpha1
kind: Application
metadata:
  name: compliance-reviewer-prod
  namespace: argocd
spec:
  project: ai-platform
  source:
    repoURL: https://github.com/fintech-corp/ai-platform-config
    targetRevision: main
    path: config/prod/compliance-reviewer
    helm:
      valueFiles:
        - values.yaml
  destination:
    server: https://kubernetes.prod.fintech-corp.internal        # <3>
    namespace: compliance-ai
  syncPolicy:
    automated:
      prune: true
      selfHeal: true
    retry:
      limit: 3
      backoff:
        duration: 30s
        factor: 2
"""

print(ARGOCD_APPLICATIONS)

1. The `repoURL` points at a **config repository** (sometimes called the "GitOps repo") that is separate from the application source repository. The config repo contains only `values.yaml` files and ArgoCD application manifests — no application code. This separation means a security team can control who has write access to production config without touching CI/CD credentials.
2. `syncPolicy.automated` with `prune: true` means ArgoCD deletes Kubernetes resources that exist in the cluster but are absent from the Git config (e.g. a manually created Secret or leftover Deployment). `selfHeal: true` means any manual `kubectl apply` or `kubectl edit` that diverges from Git is automatically reverted within 3 minutes.
3. Different `destination.server` values point at different EKS clusters — the dev application reconciles against the dev cluster, prod against the prod cluster. Both clusters have the ArgoCD operator installed; the Application CRDs themselves live in the dev cluster's ArgoCD instance and target remote clusters via `kubeconfig` credentials stored as ArgoCD Secrets.

:::{.callout-note}
The production promotion workflow is: (1) CI builds and pushes the image, tagging it with `$GITHUB_SHA`; (2) CI opens a pull request updating `config/prod/compliance-reviewer/values.yaml` with `image.tag: <new-sha>`; (3) a compliance engineer reviews and merges the PR; (4) ArgoCD detects the diff and performs a rolling update. No human touches `kubectl` or `helm` directly. The entire deployment history is the `git log` of the config repo.

:::

## Namespace Isolation and Resource Quotas

**Multi-tenant AI platform patterns.** A fintech AI platform typically serves multiple product teams — a compliance team running document reviewers, a risk team running credit scorers, a trading team running market signal classifiers. Running all of these in a single `default` namespace creates three problems: (1) a runaway pod from the trading team can exhaust node memory and evict compliance pods; (2) there is no per-team cost visibility; (3) a misconfigured NetworkPolicy on one service could expose another team's model API to cluster-internal traffic. The solution is **namespace-per-team** with `ResourceQuota`, `LimitRange`, and `NetworkPolicy`.

<br>

**Billing alignment.** `ResourceQuota` at the namespace level maps cleanly to financial chargeback. A platform team can export per-namespace resource consumption from the Kubernetes metrics API and multiply by the node cost rate to produce a monthly spend breakdown: compliance-ai: \$12,400, risk-ai: \$8,200, trading-ai: \$31,600. This is the billing model for an internal AI platform.

Complete namespace isolation manifests for the `compliance-ai` team:

In [ ]:
K8S_NAMESPACE_ISOLATION = """
# k8s/namespaces/compliance-ai.yaml
apiVersion: v1
kind: Namespace
metadata:
  name: compliance-ai
  labels:
    team:        compliance
    cost-center: ai-platform
    environment: production
---
# ResourceQuota — hard ceiling for the entire namespace
apiVersion: v1
kind: ResourceQuota
metadata:
  name: compliance-ai-quota
  namespace: compliance-ai
spec:
  hard:
    requests.cpu:            "20"             # <1>
    requests.memory:         "40Gi"
    limits.cpu:              "40"
    limits.memory:           "80Gi"
    requests.nvidia.com/gpu: "2"              # <2>
    limits.nvidia.com/gpu:   "2"
    count/pods:              "50"
    count/services:          "10"
    count/persistentvolumeclaims: "20"
---
# LimitRange — default requests/limits for pods that don't specify them
apiVersion: v1
kind: LimitRange
metadata:
  name: compliance-ai-limits
  namespace: compliance-ai
spec:
  limits:
    - type: Container
      default:                                # <3>
        cpu:    "500m"
        memory: "512Mi"
      defaultRequest:
        cpu:    "100m"
        memory: "128Mi"
      max:
        cpu:    "8"
        memory: "16Gi"
      min:
        cpu:    "50m"
        memory: "64Mi"
---
# NetworkPolicy — deny all ingress/egress by default, then allow selectively
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: compliance-ai-default-deny
  namespace: compliance-ai
spec:
  podSelector: {}                             # <4>
  policyTypes:
    - Ingress
    - Egress
---
# Allow ingress from the API gateway namespace only
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: compliance-ai-allow-gateway
  namespace: compliance-ai
spec:
  podSelector:
    matchLabels:
      app: compliance-reviewer
  policyTypes:
    - Ingress
  ingress:
    - from:
        - namespaceSelector:
            matchLabels:
              kubernetes.io/metadata.name: api-gateway   # <5>
      ports:
        - protocol: TCP
          port: 8000
---
# Allow egress to AWS endpoints and the SQS queue, nothing else
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: compliance-ai-allow-aws-egress
  namespace: compliance-ai
spec:
  podSelector: {}
  policyTypes:
    - Egress
  egress:
    - ports:
        - protocol: TCP
          port: 443                           # <6>
    - ports:
        - protocol: TCP
          port: 5432                          # PostgreSQL (RDS)
"""

print(K8S_NAMESPACE_ISOLATION)

1. `requests.cpu: "20"` means the sum of CPU requests across all pods in `compliance-ai` cannot exceed 20 cores. The scheduler enforces this: a new pod whose request would push the total over the quota is rejected with `exceeded quota`. The quota is set to the compliance team's allocated share of the cluster, enabling chargeback.
2. GPU quotas are enforced in the same quota object. Without `requests.nvidia.com/gpu: "2"`, a misconfigured deployment could consume all GPU nodes in the cluster, starving other teams' workloads.
3. `LimitRange.default` sets limits for containers that omit them entirely — preventing a pod with no limit from consuming unbounded memory and triggering OOM kills on the node. The `min` field rejects pods with unrealistically low requests that would cause over-scheduling.
4. `podSelector: {}` (empty) matches all pods in the namespace. The default-deny policy is the starting point: all traffic is blocked. Subsequent NetworkPolicy resources then carve out the minimum necessary exceptions.
5. `namespaceSelector` allows ingress only from pods in the `api-gateway` namespace. Pods in `risk-ai` or `trading-ai` cannot reach the compliance reviewer directly — all cross-team traffic must flow through the API gateway, which enforces authentication and rate limiting.
6. Port 443 covers HTTPS egress to AWS services (SQS, Secrets Manager, S3, CloudWatch) and the OpenAI API. The `compliance-reviewer` pod has no legitimate reason to open a connection to any non-HTTPS port on the public internet.

## Cost Governance

**The GPU cost problem.** A single `g4dn.xlarge` instance costs \$0.526/hr on-demand. A GPU cluster of 10 nodes running continuously costs $46,000/month — before any data transfer, storage, or API fees. For an AI inference platform processing bursty, event-driven workloads (compliance documents arrive in batches at market close), the cluster is idle for large windows every day. Three cost levers matter: (1) spot instances for interruptible workloads; (2) Savings Plans for baseline always-on compute; and (3) Karpenter for bin-packing — eliminating the gap between requested and allocated resources that Cluster Autoscaler cannot close.

<br>

**Savings Plans vs. Reserved Instances vs. On-Demand vs. Spot.** The choice is a function of utilisation fraction $u$ (the fraction of time the node is actually needed) and the commitment duration $T$ (months). The savings from switching from on-demand to a pricing tier is:

$$\text{savings} = (p_{\text{OD}} - p_{\text{tier}}) \cdot u \cdot T$$

where $p_{\text{OD}}$ is the on-demand hourly rate and $p_{\text{tier}}$ is the committed or spot rate. For a `g4dn.xlarge` with $u = 1.0$ (always-on inference endpoint), a 1-year Compute Savings Plan at \$0.319/hr breaks even in $T = 0$ months (immediately cheaper). For $u = 0.3$ (batch GPU job running 7h/day), the annualized savings are $(0.526 - 0.319) \times 0.3 \times 8760 \approx \$544$ per node per year — still positive, but a spot instance at $\approx$\$0.16/hr dominates at $u < 0.6$ assuming the workload tolerates interruption.

Spot instance tier comparison for GPU inference nodes:

In [ ]:
import math

# g4dn.xlarge pricing (us-east-1, approximate)
p_od   = 0.526   # on-demand $/hr
p_sp1y = 0.319   # 1-year Compute Savings Plan $/hr
p_ri3y = 0.208   # 3-year Reserved Instance (all-upfront) $/hr
p_spot = 0.158   # spot (approximate average, interruptible)

T_months = 12    # analysis horizon
T_hours  = T_months * 730  # ~730 hr/month

print(f"g4dn.xlarge cost comparison over {T_months} months (10 nodes)")
print(f"{'Tier':<30} {'$/hr':>8} {'Annual $/node':>16} {'vs. OD':>10}")
print("-" * 68)

for label, price in [
    ("On-Demand",                     p_od),
    ("1-yr Savings Plan",             p_sp1y),
    ("3-yr Reserved (all-upfront)",   p_ri3y),
    ("Spot (avg, u=1.0)",             p_spot),
]:
    annual = price * T_hours
    saving_pct = (p_od - price) / p_od * 100
    print(f"{label:<30} {price:>8.3f} {annual:>16,.0f}  {saving_pct:>9.1f}%")

print()
print("Breakeven utilisation for Savings Plan vs. Spot:")
# savings(SP) = (p_od - p_sp) * u * T
# savings(Spot) = (p_od - p_spot) * u * T  (but spot has interruption risk)
# SP is preferred when workload cannot tolerate interruption
for u in [0.2, 0.4, 0.6, 0.8, 1.0]:
    savings_sp   = (p_od - p_sp1y) * u * T_hours
    savings_spot = (p_od - p_spot) * u * T_hours
    print(f"  u={u:.1f}: SP saves ${savings_sp:,.0f}/yr | Spot saves ${savings_spot:,.0f}/yr per node")

**Spot interruption handling.** AWS gives a 2-minute warning before reclaiming a spot instance. The `aws-node-termination-handler` daemonset intercepts this event and cordons the node (preventing new pods from scheduling on it), drains existing pods gracefully via SIGTERM, and waits for `terminationGracePeriodSeconds` before force-killing. A `PodDisruptionBudget` (PDB) ensures the drain does not take more than one GPU pod offline simultaneously — protecting the service SLA during reclaims:

In [ ]:
K8S_SPOT_RESILIENCE = """
# k8s/pdb.yaml — PodDisruptionBudget for GPU inference
apiVersion: policy/v1
kind: PodDisruptionBudget
metadata:
  name: embeddings-server-pdb
  namespace: risk-ai
spec:
  minAvailable: 1                     # <1>
  selector:
    matchLabels:
      app: embeddings-server
---
# Graceful shutdown in the application (Dockerfile CMD or Python entrypoint)
# The container must handle SIGTERM and drain in-flight requests:
#
# import signal, sys
#
# def handle_sigterm(sig, frame):
#     print('{"level":"info","msg":"SIGTERM received, draining"}')  # <2>
#     app.state.shutdown = True      # stop accepting new requests
#     time.sleep(15)                 # wait for in-flight LLM calls to complete
#     sys.exit(0)
#
# signal.signal(signal.SIGTERM, handle_sigterm)
---
# terminationGracePeriodSeconds must be > the longest expected LLM call
# For gpt-4o-mini with 2k output tokens: ~15s
# For local Llama-3-8B on g4dn.xlarge: ~45s
# Set in the Deployment spec:
# spec.template.spec.terminationGracePeriodSeconds: 60          # <3>
"""

print(K8S_SPOT_RESILIENCE)

1. `minAvailable: 1` means the drain operation will pause if removing a pod would leave zero pods running. For a single-GPU service with two replicas, draining one pod during a spot reclaim leaves one pod available — the SLA degrades but does not break.
2. Structured JSON logging of the shutdown event is essential for post-mortem analysis: it tells operators whether a pod terminated due to a spot reclaim (SIGTERM) or an application crash (non-zero exit code).
3. `terminationGracePeriodSeconds` must be greater than the slowest in-flight request. For a GPU inference server running Llama-3-8B, a 45-second generation timeout plus 15 seconds buffer gives 60 seconds. The `aws-node-termination-handler` blocks the node drain until this period elapses.

A Karpenter `NodePool` (formerly `Provisioner`) that bin-packs GPU pods just-in-time:

In [ ]:
KARPENTER_NODEPOOL = """
# k8s/karpenter/gpu-nodepool.yaml
apiVersion: karpenter.sh/v1
kind: NodePool
metadata:
  name: gpu-inference
spec:
  template:
    metadata:
      labels:
        role: gpu-inference
    spec:
      requirements:
        - key: karpenter.sh/capacity-type
          operator: In
          values: ["spot", "on-demand"]       # <1>
        - key: node.kubernetes.io/instance-type
          operator: In
          values:
            - g4dn.xlarge
            - g4dn.2xlarge
            - g5.xlarge
            - g5.2xlarge                      # <2>
        - key: topology.kubernetes.io/zone
          operator: In
          values: ["us-east-1a", "us-east-1b", "us-east-1c"]
      nodeClassRef:
        apiVersion: karpenter.k8s.aws/v1
        kind: EC2NodeClass
        name: gpu-node-class
      taints:
        - key: nvidia.com/gpu
          value: "true"
          effect: NoSchedule
  limits:
    nvidia.com/gpu: 20                        # <3>
  disruption:
    consolidationPolicy: WhenEmptyOrUnderutilized  # <4>
    consolidateAfter: 1m
---
apiVersion: karpenter.k8s.aws/v1
kind: EC2NodeClass
metadata:
  name: gpu-node-class
spec:
  amiFamily: AL2
  role: eks-node-role
  subnetSelectorTerms:
    - tags:
        karpenter.sh/discovery: ml-platform-prod
  securityGroupSelectorTerms:
    - tags:
        karpenter.sh/discovery: ml-platform-prod
  instanceStorePolicy: RAID0               # <5>
"""

print(KARPENTER_NODEPOOL)

1. Karpenter tries spot first; if no spot capacity is available across the listed instance types and zones, it falls back to on-demand. This is more resilient than a pure-spot node group that simply fails to provision when spot is unavailable.
2. Allowing multiple GPU instance families (`g4dn`, `g5`) increases the spot capacity pool, dramatically reducing the probability of simultaneous unavailability across all types — the key to reliable spot-based GPU inference.
3. `limits.nvidia.com/gpu: 20` is a hard cap on the total number of GPUs this NodePool can provision. This prevents a misconfigured KEDA scaler from spinning up an unbounded number of GPU nodes and generating a five-figure bill before anyone notices.
4. `consolidationPolicy: WhenEmptyOrUnderutilized` instructs Karpenter to terminate nodes when their pods can be bin-packed onto fewer nodes. The Cluster Autoscaler cannot do this — it only scales down fully empty nodes. Karpenter's consolidation is the primary driver of the 20–40% cost reduction over Cluster Autoscaler that teams report in practice.
5. `instanceStorePolicy: RAID0` configures the NVMe instance store SSDs on GPU instances as a RAID-0 array, providing fast local scratch space for model weight caching — loading a 7B parameter model from local NVMe takes seconds vs. minutes from S3.

:::{.callout-caution}
Never use `consolidationPolicy: WhenEmptyOrUnderutilized` without a `PodDisruptionBudget` for every GPU workload. Karpenter's consolidation drains pods to repack nodes — if a GPU model server has no PDB, a consolidation event can take it offline mid-inference, returning a 502 to the upstream API caller.

:::

A Kubecost label taxonomy and cost allocation query illustrating per-team spend visibility:

In [ ]:
# Simulate Kubecost-style cost allocation from pod resource consumption
# In production, Kubecost queries the Kubernetes metrics API and node pricing
# and exposes /api/v1/allocation for per-namespace/per-label breakdowns.

# g4dn.xlarge: 4 vCPU, 16 GiB RAM, 1 GPU  @ $0.526/hr on-demand
NODE_CPU_CORES   = 4
NODE_MEMORY_GiB  = 16
NODE_GPUS        = 1
NODE_COST_HR     = 0.526

# Cost per unit per hour (RAM-weighted allocation)
CPU_COST_CORE_HR = NODE_COST_HR * 0.40 / NODE_CPU_CORES    # 40% of node cost to CPU
MEM_COST_GiB_HR  = NODE_COST_HR * 0.30 / NODE_MEMORY_GiB  # 30% to memory
GPU_COST_HR      = NODE_COST_HR * 0.30 / NODE_GPUS         # 30% to GPU

# Pod resource requests per team namespace (monthly, 730 hr)
HOURS = 730
teams = {
    "compliance-ai": dict(cpu_cores=8,  mem_GiB=16, gpus=0, pods=12),
    "risk-ai":       dict(cpu_cores=12, mem_GiB=24, gpus=1, pods=8),
    "trading-ai":    dict(cpu_cores=32, mem_GiB=64, gpus=4, pods=20),
}

print(f"{'Namespace':<20} {'CPU':>8} {'Memory':>10} {'GPU':>8} {'Total/mo':>12} {'Efficiency':>12}")
print("-" * 74)
total = 0
for ns, r in teams.items():
    cpu_cost = r["cpu_cores"] * CPU_COST_CORE_HR * HOURS
    mem_cost = r["mem_GiB"]  * MEM_COST_GiB_HR  * HOURS
    gpu_cost = r["gpus"]     * GPU_COST_HR       * HOURS
    ns_total = cpu_cost + mem_cost + gpu_cost
    # Efficiency: requested / (allocated node capacity)
    # Simplified: pods * avg_utilisation vs. node ceiling
    efficiency = min(1.0, r["pods"] / (r["pods"] * 1.25)) * 100
    total += ns_total
    print(f"{ns:<20} ${cpu_cost:>6,.0f} ${mem_cost:>8,.0f} ${gpu_cost:>6,.0f} ${ns_total:>10,.0f}   {efficiency:>8.0f}%")
print("-" * 74)
print(f"{'Total':<20} {'':>8} {'':>10} {'':>8} ${total:>10,.0f}")

## Production Checklist

A 20-item readiness checklist for an AI/ML workload going to production, organised into four categories.

<br>

**IaC hygiene**

1. **Remote state with locking.** S3 backend with a DynamoDB lock table. No local `terraform.tfstate` files in version control — ever.
2. **State encryption.** S3 bucket has `server_side_encryption_configuration` with AES-256 or KMS. The state file contains sensitive resource attributes (IAM ARNs, security group IDs).
3. **Module versioning.** Internal modules are pinned to a Git tag (`source = "git::https://...?ref=v1.3.0"`), not `ref=main`. Unpinned modules break on the next upstream commit.
4. **Drift detection in CI.** A scheduled GitHub Actions job runs `terraform plan` daily and posts the diff to Slack. Unintended drift (manual console changes, expiring resources) is caught before it causes an incident.
5. **Workspace-per-environment.** Dev, staging, and prod use separate workspace state. A `terraform apply` in staging cannot modify prod infrastructure regardless of variable values.

<br>

**Kubernetes hardening**

6. **Resource requests and limits on every pod.** No pod in production has unspecified `requests` or `limits`. Enforce with OPA Gatekeeper or Kyverno admission webhook.
7. **Non-root containers.** All containers run as `runAsUser: 1000` with `runAsNonRoot: true`. Verify with `kubectl get pods -o jsonpath='{.spec.containers[*].securityContext}'`.
8. **Read-only root filesystem.** `readOnlyRootFilesystem: true` on all containers. Application writes go to explicitly mounted `emptyDir` volumes — this limits blast radius of a container escape.
9. **Network policies.** Every namespace has a default-deny ingress and egress policy. Each service has explicit allow policies for the minimum necessary traffic.
10. **RBAC least privilege.** Service accounts have only the IAM permissions they need (IRSA). No pod runs with the node's instance profile. `kubectl auth can-i --list --as=system:serviceaccount:compliance-ai:default` should return a short list.

<br>

**Observability**

11. **Structured JSON logs.** Every service emits logs as `{"level": "info", "msg": "...", "trace_id": "..."}`. No `print()` statements in production code. CloudWatch Logs Insights can then query by field.
12. **CloudWatch EMF metrics.** Latency, token cost, error rate, and cache hit rate emitted via Embedded Metric Format. No synchronous `PutMetricData` calls in the request path.
13. **Distributed tracing.** AWS X-Ray sidecar or OpenTelemetry collector in each pod. Trace propagation headers (`X-Amzn-Trace-Id`) passed through to the LLM provider call. End-to-end latency breakdown for every slow request.
14. **SLO alerting.** CloudWatch Alarms fire when P99 latency exceeds the SLO, error rate exceeds 1%, or monthly LLM spend exceeds the budget. Alarms route to PagerDuty (latency/errors) and email (cost).
15. **Kubecost or OpenCost.** Per-namespace cost allocation visible to each team daily. Idle cost and cluster efficiency score tracked weekly.

<br>

**Compliance and security**

16. **Immutable audit log for deployments.** GitOps (ArgoCD) means every deployment is a Git commit with a SHA, author, PR number, and approver. The config repository is append-only (branch protection with no force-push).
17. **Secrets in Secrets Manager, never in environment variables.** ECS secrets injection or Kubernetes External Secrets Operator syncs secrets into pods. Running `kubectl describe pod` must not reveal secret values — only the secret reference name.
18. **ECR image scanning.** `scan_on_push = true` on all ECR repositories. CI pipeline fails if a `CRITICAL` severity finding is present in the scanned image. Use `aws ecr describe-image-scan-findings` in the CI gate step.
19. **IAM least privilege for CI/CD.** The GitHub Actions OIDC role has `ecr:GetAuthorizationToken`, `ecr:BatchCheckLayerAvailability`, `ecr:PutImage` — and nothing else. It cannot modify ECS services, read Secrets Manager, or access S3. The deploy step assumes a separate role with ECS permissions.
20. **PodDisruptionBudgets for all stateful or GPU workloads.** Every deployment with `minReplicas ≥ 2` that serves real traffic has a PDB with `minAvailable: 1`. Node drain events (upgrades, spot reclaims, Karpenter consolidation) cannot take a service to zero replicas without explicit operator intervention.

:::{.callout-important}
Items 17 (secrets in Secrets Manager) and 19 (least-privilege CI/CD role) are the two most commonly violated items in fintech AI platform reviews. A `kubectl describe pod` that shows `OPENAI_API_KEY=sk-...` in the environment output, or a CI role with `iam:*` permissions, is an automatic fail in a SOC 2 audit.

:::

## S3 as an AI Data Store

Notebook 12 uses S3 only as the Terraform remote-state backend. In practice, S3 is the backbone of every AI/ML data layer on AWS: raw documents live in S3, fine-tuning datasets are uploaded there, embedding snapshots are persisted there, and model artefacts are versioned there. The patterns are straightforward but the details — prefix design, lifecycle rules, presigned URLs for secure access, and versioning for reproducibility — are routinely asked in design interviews.

The key S3 concepts for an AI platform:

**Bucket layout.** A common convention organises data by lifecycle stage:
```
s3://fintech-ai-{env}/
  raw/documents/          # ingested source docs (SEC filings, PDFs)
  processed/chunks/       # chunked + normalised text
  embeddings/snapshots/   # serialised numpy arrays or ChromaDB exports
  models/fine-tuned/      # LoRA adapters, GGUF weights
  eval/datasets/          # golden eval datasets
  eval/results/           # eval run outputs (JSON)
```

**Versioning.** Enable S3 versioning on the bucket so every overwrite is recoverable. For eval datasets this is critical: you need to reproduce a past eval run against the exact dataset version that was used when the model was approved.

**Lifecycle rules.** Move raw ingest (accessed once) to S3-IA after 30 days, then to Glacier after 90. Keep processed chunks in Standard (accessed on every index rebuild). Expire old eval results after 1 year unless compliance requires longer.

**Presigned URLs.** Never expose your S3 bucket publicly. Generate short-lived presigned URLs (15 minutes) for the FastAPI service to return document download links to authenticated users. This keeps IAM credentials server-side.

We implement an `AIDataStore` class that wraps `boto3` with the bucket layout above. Because this is a conceptual notebook (no live AWS account), we use `moto` — a mock AWS library — to run the same `boto3` calls against an in-process fake S3.

In [ ]:
import io, json as _json, datetime
import boto3
from moto import mock_aws  # <1>

BUCKET = "fintech-ai-staging"

class AIDataStore:  # <2>
    """Thin wrapper around S3 for the fintech AI data layer."""

    def __init__(self, bucket: str, s3_client=None):
        self.bucket = bucket
        self.s3 = s3_client or boto3.client("s3", region_name="us-east-1")

    # --- documents ---
    def upload_document(self, doc_id: str, content: bytes, content_type="application/pdf") -> str:  # <3>
        key = f"raw/documents/{doc_id}"
        self.s3.put_object(Bucket=self.bucket, Key=key, Body=content, ContentType=content_type)
        return key

    def presigned_url(self, key: str, expiry_seconds: int = 900) -> str:  # <4>
        return self.s3.generate_presigned_url(
            "get_object",
            Params={"Bucket": self.bucket, "Key": key},
            ExpiresIn=expiry_seconds,
        )

    # --- embeddings ---
    def save_embedding_snapshot(self, snapshot_id: str, data: bytes) -> str:
        key = f"embeddings/snapshots/{snapshot_id}.npy"
        self.s3.put_object(Bucket=self.bucket, Key=key, Body=data)
        return key

    # --- eval datasets ---
    def save_eval_dataset(self, run_id: str, records: list) -> str:
        key = f"eval/datasets/{run_id}.jsonl"
        body = "\n".join(_json.dumps(r) for r in records).encode()
        self.s3.put_object(Bucket=self.bucket, Key=key, Body=body)
        return key

    def load_eval_dataset(self, run_id: str) -> list:
        key = f"eval/datasets/{run_id}.jsonl"
        obj = self.s3.get_object(Bucket=self.bucket, Key=key)
        return [_json.loads(line) for line in obj["Body"].read().decode().splitlines()]

    def list_prefix(self, prefix: str) -> list:
        resp = self.s3.list_objects_v2(Bucket=self.bucket, Prefix=prefix)
        return [o["Key"] for o in resp.get("Contents", [])]


**1.** `mock_aws` from `moto` intercepts all `boto3` calls and routes them to an in-process fake AWS environment — no credentials or live account required. **2.** `AIDataStore` encapsulates the bucket layout; callers never construct S3 keys manually. **3.** `upload_document` returns the S3 key so callers can store it in the metadata database. **4.** `presigned_url` generates a time-limited URL — the default 15-minute window is appropriate for a single download triggered by an API response.

We run the store inside a `mock_aws` context. All S3 operations behave identically to real AWS.

In [ ]:
import numpy as np

@mock_aws
def demo_s3():
    s3 = boto3.client("s3", region_name="us-east-1")
    s3.create_bucket(Bucket=BUCKET)
    store = AIDataStore(bucket=BUCKET, s3_client=s3)

    # Upload a synthetic SEC filing
    key = store.upload_document("AAPL-10K-2024.pdf", b"%PDF synthetic content", "application/pdf")
    url = store.presigned_url(key)
    print(f"Uploaded: {key}")
    print(f"Presigned URL (truncated): {url[:80]}...")

    # Save an embedding snapshot
    vecs = np.random.randn(20, 1536).astype("float32")
    buf = io.BytesIO()
    np.save(buf, vecs)
    snap_key = store.save_embedding_snapshot("sec-corpus-v1", buf.getvalue())
    print(f"\nEmbedding snapshot: {snap_key}")

    # Save and reload an eval dataset
    records = [{"question": f"Q{i}", "answer": f"A{i}"} for i in range(5)]
    ds_key = store.save_eval_dataset("golden-v1", records)
    loaded = store.load_eval_dataset("golden-v1")
    print(f"\nEval dataset: {ds_key} ({len(loaded)} records)")

    # List all keys under raw/
    keys = store.list_prefix("raw/")
    print(f"\nraw/ prefix contents: {keys}")

demo_s3()


## Observability: OpenTelemetry and Prometheus

Notebook 12 covers CloudWatch EMF for metrics and Langfuse for LLM-specific tracing. This section covers the complementary open-standards stack: **OpenTelemetry** (OTEL) for distributed tracing and **Prometheus** for metrics scraping. Both are standard in Kubernetes environments and appear in fintech principal-engineer interviews.

**Why OTEL alongside Langfuse?** Langfuse traces LLM calls (prompts, tokens, cost). OTEL traces the *service* layer: HTTP request → queue dequeue → preprocessing → LLM call → postprocessing → response. Together they give end-to-end visibility from the user's HTTP request down to the individual LLM token generation.

**Why Prometheus alongside CloudWatch?** CloudWatch is AWS-proprietary and billed per metric. Prometheus is open-source, runs in-cluster, and is the standard for Kubernetes workloads. In practice, you use both: Prometheus for in-cluster alerting (zero egress cost), CloudWatch for cross-account aggregation and executive dashboards.

**OTEL instrumentation pattern for a FastAPI service:**
```python
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.instrumentation.fastapi import FastAPIInstrumentor

provider = TracerProvider()
provider.add_span_processor(
    BatchSpanProcessor(OTLPSpanExporter(endpoint="http://otel-collector:4318/v1/traces"))
)
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("compliance-reviewer")

# Auto-instrument FastAPI (injects trace/span context into every request)
FastAPIInstrumentor.instrument_app(app)

# Manual span for LLM call
with tracer.start_as_current_span("llm.complete") as span:
    span.set_attribute("llm.model", model)
    span.set_attribute("llm.input_tokens", n_input)
    result = llm.complete(messages)
    span.set_attribute("llm.output_tokens", n_output)
```
The OTEL Collector (deployed as a DaemonSet or sidecar) receives spans and fans them out to Jaeger (local UI), Tempo (Grafana backend), or AWS X-Ray (production).

**Prometheus metrics for an LLM service.** We implement a `PrometheusMetrics` class that exposes the four Golden Signals as Prometheus counters and histograms. In a real deployment these would be scraped by Prometheus via the `/metrics` endpoint; here we simulate the metric registration and show the exposition format.

In [ ]:
from prometheus_client import Counter, Histogram, Gauge, generate_latest, REGISTRY  # <1>
import random, time

# Clear any previously registered metrics (notebook re-run safety)
collectors = list(REGISTRY._names_to_collectors.keys())
for name in ['llm_requests_total', 'llm_latency_seconds', 'llm_tokens_total', 'llm_errors_total']:
    if name in collectors:
        REGISTRY.unregister(REGISTRY._names_to_collectors[name])

# --- Four Golden Signals for an LLM service ---
REQUEST_COUNT = Counter(  # <2>
    "llm_requests_total",
    "Total LLM API requests",
    ["model", "endpoint", "status"],
)

LATENCY = Histogram(  # <3>
    "llm_latency_seconds",
    "LLM request latency in seconds",
    ["model", "endpoint"],
    buckets=[0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0],
)

TOKEN_COUNT = Counter(
    "llm_tokens_total",
    "Cumulative tokens consumed",
    ["model", "token_type"],  # token_type: input | output
)

ERROR_COUNT = Counter(
    "llm_errors_total",
    "LLM API error count",
    ["model", "error_type"],
)

class InstrumentedLLMService:  # <4>
    """Wraps LLMClient with Prometheus instrumentation."""

    def __init__(self, model="gpt-4o-mini"):
        self.model = model

    def review(self, text: str, endpoint: str = "compliance") -> dict:
        start = time.perf_counter()
        try:
            # Simulate LLM call (replace with real llm.complete() in production)
            time.sleep(random.uniform(0.05, 0.3))
            if random.random() < 0.05:
                raise RuntimeError("rate_limit")
            n_in, n_out = len(text.split()), random.randint(20, 80)
            TOKEN_COUNT.labels(model=self.model, token_type="input").inc(n_in)
            TOKEN_COUNT.labels(model=self.model, token_type="output").inc(n_out)
            REQUEST_COUNT.labels(model=self.model, endpoint=endpoint, status="ok").inc()
            return {"result": "compliant", "tokens": n_in + n_out}
        except Exception as exc:
            ERROR_COUNT.labels(model=self.model, error_type=str(exc)).inc()
            REQUEST_COUNT.labels(model=self.model, endpoint=endpoint, status="error").inc()
            return {"result": "error", "error": str(exc)}
        finally:
            LATENCY.labels(model=self.model, endpoint=endpoint).observe(
                time.perf_counter() - start
            )

svc = InstrumentedLLMService()
docs = [
    "Section 5.3: margin requirement shall not exceed 50% of notional.",
    "Client may withdraw funds subject to 48-hour notice period.",
    "Interest rate risk hedged via IRS per ISDA master agreement.",
]
for _ in range(20):
    svc.review(random.choice(docs))

# Print Prometheus text exposition format
output = generate_latest(REGISTRY).decode()  # <5>
for line in output.splitlines():
    if line and not line.startswith('#'):
        print(line)


**1.** `prometheus_client` is the Python client for Prometheus. In a FastAPI service, mount it as `GET /metrics` using `make_asgi_app()` — Prometheus scrapes this endpoint on a configurable interval. **2.** `Counter` is for monotonically increasing values (request counts, error counts, token totals). **3.** `Histogram` automatically tracks count, sum, and configurable quantile buckets; the `le` label on each bucket lets Prometheus compute $p_{50}$, $p_{95}$, $p_{99}$ latency with a PromQL query. **4.** `InstrumentedLLMService` wraps the LLM call in a try/finally that always records latency, ensuring even errored requests are tracked. **5.** `generate_latest()` serialises all registered metrics in the Prometheus text exposition format — exactly what Prometheus scrapes from `/metrics`.

**Grafana dashboard and alerting.** Once Prometheus scrapes `/metrics`, standard PromQL queries power a Grafana dashboard:

| Panel | PromQL |
|---|---|
| Request rate | `rate(llm_requests_total[5m])` |
| Error rate | `rate(llm_errors_total[5m]) / rate(llm_requests_total[5m])` |
| p99 latency | `histogram_quantile(0.99, rate(llm_latency_seconds_bucket[5m]))` |
| Token burn rate | `rate(llm_tokens_total[1h]) * 3600` |

**Alerting rule example** (Prometheus `rules.yaml`):
```yaml
groups:
  - name: llm-slo
    rules:
      - alert: LLMHighErrorRate
        expr: |
          rate(llm_errors_total[5m])
          / rate(llm_requests_total[5m]) > 0.01
        for: 2m
        labels:
          severity: critical
        annotations:
          summary: "LLM error rate above 1% SLO"
```

:::{.callout-tip}
In a Kubernetes deployment, add `prometheus.io/scrape: 'true'` and `prometheus.io/port: '8000'` annotations to the pod template. The Prometheus Kubernetes SD (service discovery) will automatically pick up the endpoint without any manual scrape config.

:::

---

$\blacksquare$